In [16]:
import numpy as np

In [17]:
# Creating a wrapper around numerical values
class Value:
    def __init__(self, data):
        self.data = data

In [18]:
a = Value(2.0)
b = Value(3.0)

print(a.data)
print(b.data)

2.0
3.0


In [19]:
# The below returns an error message
# a + b

- `__add__(self, other) (Special / Magic Method)`

- How it's called: Automatically triggered whenever you use the plus operator (+) between two objects (e.g., a + b or a + 2). Python behind the scenes translates a + b into a.__add__(b).

- `Syntax Benefits`: It allows you to write natural, readable mathematical expressions and chain operations seamlessly (e.g., a + b * c).

- In your code: Because of __add__, you can do:

---

- `add(self, other) (Regular Method)`
- How it's called: Must be called explicitly like any standard Python method using dot notation (e.g., a.add(b)).

- Syntax Impact: It does not hook into Python's built-in operators. You cannot use the + symbol with it; doing so would raise a TypeError.


In [20]:
# Add Operations
# Now we want to support addition and multipliation

class Value:
    def __init__(self, data):
        self.data = data

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data)

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data)

This code defines a custom `Value` class, which is the foundational building block for creating a **miniature automatic differentiation (autograd) engine** from scratch. This exact pattern is famously used in Andrej Karpathy's *micrograd* library to teach how neural networks work under the hood.

Here is a step-by-step breakdown of what is happening inside the class:

---

### 1. Initialization (`__init__`)

```python
def __init__(self, data):
    self.data = data

```

* **What it does:** When you create a new `Value` object (e.g., `x = Value(4.0)`), this method runs and stores your number inside an attribute called `self.data`.

### 2. Addition Operator Overloading (`__add__`)

```python
def __add__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    return Value(self.data + other.data)

```

* **What it does:** This intercepts the `+` operator (e.g., `a + b`).
* **The smart type-checking line (`isinstance`):** In Python, if you try to add `Value(2) + 3`, Python would normally throw a `TypeError` because `3` doesn't have a `.data` attribute. This line checks if `other` is already a `Value` object. If it's just a raw number (`int` or `float`), it automatically wraps it into a `Value(3)` object so your code doesn't crash.
* **The result:** It extracts the raw numbers from both objects (`self.data` and `other.data`), adds them together, and wraps the resulting sum inside a **brand new `Value` object**.

### 3. Multiplication Operator Overloading (`__mul__`)

```python
def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    return Value(self.data * other.data)

```

* **What it does:** This works identically to `__add__`, but for the multiplication operator (`*`).
* **The result:** It multiplies the underlying numbers together and returns the product wrapped in a new `Value` object.

---

### Putting It All Together in Action

Because of these "magic methods," you can write mathematical expressions that look and feel like normal Python math, even though they are operating on custom objects:

```python
a = Value(2)
b = Value(3)

# Python automatically calls a.__add__(b)
c = a + b  
print(c.data)  # Output: 5

# You can even mix Values and raw numbers thanks to the isinstance check:
d = c * 4  # Python automatically calls c.__mul__(4)
print(d.data)  # Output: 20

```

In [21]:
# Now we can do the below
a = Value(2.0)
b = Value(3.0)

c = a + b
d = c * b
print(d.data)


15.0


In [22]:
# We can see the below
print(a.data)
print(b.data)
print(c.data)
print(d.data)

2.0
3.0
5.0
15.0


#### Notice the following Python methods:
- __add__: allows us to use +.
- __mul__: allows us to use *.

*These are called operator overloading methods.*

### Extend the Value Object to remember the computational history

In [23]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0

        self._prev = set(_children)
        self._op = _op
        self.label = ""

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)

        return Value(
            self.data + other.data,
            (self, other),
            "+"
        )

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)

        return Value(
            self.data * other.data,
            (self, other),
            "*"
        )


In [24]:
# testing the above 
a = Value(2.0)
b = Value(3.0)
c = a + b
d = c*b

In [25]:
print(a.data)
print(a._prev)
print(a._op)

2.0
set()



In [26]:
print(b.data)
print(b._prev)
print(b._op)

3.0
set()



In [27]:
print(c.data)
print(c._prev)
print(c._op)

5.0
{<__main__.Value object at 0x778839722330>, <__main__.Value object at 0x778839722fc0>}
+


**The important new attributes are:**
- Attribute	Purpose
- data	Numerical value
- grad	Gradient
- prev	Previous nodes
- op	Operation used
- label	Optional descriptive name

In [28]:
print(d.data)
print(d._prev)
print(d._op)

15.0
{<__main__.Value object at 0x778839722fc0>, <__main__.Value object at 0x77883b5a0fe0>}
*


### Note: when a variable contributes to an output through multiple paths, accumulae teh gradient from all paths.

In [29]:

a = 2.0
b = 3.0

c = a + b
d = c * b

# Gradients calculated manually
d_grad = 1.0

c_grad = b * d_grad
b_grad_direct = c * d_grad

a_grad = c_grad * 1.0
b_grad_indirect = c_grad * 1.0

b_grad = b_grad_direct + b_grad_indirect

print("Output:", d)
print("Gradient of a:", a_grad)
print("Gradient of b:", b_grad)


Output: 15.0
Gradient of a: 3.0
Gradient of b: 8.0


In [30]:
c_grad = b * d_grad
